# CNN

Notebook ini mencakup Bagian 1 sampai Bagian 4: utility image processing, feature extraction, training 16 arsitektur Conv2D shared parameter, evaluasi macro F1-score, dan perbandingan Keras vs forward propagation from scratch.


In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "cnn":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from scripts.run_cnn_experiments import config_from_yaml, generate_shared_conv_grid, run_grid
from tubes2_ml.cnn.data import load_image, load_image_batch
from tubes2_ml.cnn.evaluate import build_intel_test_dataset, evaluate_keras_model, evaluate_scratch_model
from tubes2_ml.cnn.feature_extraction import extract_features_to_npy
from tubes2_ml.cnn.train import configure_tensorflow_runtime
from tubes2_ml.scratch.models.cnn_classifier import build_scratch_cnn_from_keras
from tubes2_ml.visualization.feature_maps import save_conv_feature_visualizations
from tubes2_ml.visualization.grad_cam import save_gradcam

CONFIG_PATH = PROJECT_ROOT / "configs" / "cnn" / "shared_conv.yaml"
runtime_devices = configure_tensorflow_runtime()
model_config, training_config = config_from_yaml(CONFIG_PATH)
model_config, training_config


## Bagian 1: Utility Functions

Cell berikut memakai `PIL/Pillow` dan `NumPy` untuk loader image, batch loader, dan feature extractor ke file `.npy`. Feature extractor memakai Keras CNN encoder yang dibekukan.


In [ ]:
sample_image_paths = sorted((PROJECT_ROOT / "data/raw/intel_image_classification/seg_test/seg_test").glob("*/*.jpg"))[:8]

sample_image = load_image(sample_image_paths[0], target_size=training_config.image_size)
sample_batch = load_image_batch(sample_image_paths, target_size=training_config.image_size)

print("Single image shape:", sample_image.shape, "range:", (float(sample_image.min()), float(sample_image.max())))
print("Batch shape:", sample_batch.shape)


In [ ]:
feature_encoder = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=model_config.input_shape),
        tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.GlobalAveragePooling2D(),
    ],
    name="frozen_cnn_encoder",
)
feature_encoder.trainable = False

feature_path = PROJECT_ROOT / "data/features/cnn/sample_features.npy"
sample_features = extract_features_to_npy(
    image_paths=sample_image_paths,
    encoder=feature_encoder,
    output_path=feature_path,
    target_size=training_config.image_size,
    batch_size=4,
    overwrite=False,
)

print("Feature shape:", sample_features.shape)
print("Saved to:", feature_path)


## Bagian 2: Forward Propagation From Scratch

Implementasi layer scratch ada di `src/tubes2_ml/scratch/layers`: `Conv2D`, `LocallyConnected2D`, pooling, global pooling, flatten, dense, dan aktivasi. Builder `build_scratch_cnn_from_keras` membaca bobot model Keras dan menyusun forward propagation NumPy.


## Bagian 3: Pelatihan Model

Training tetap menjalankan 16 variasi arsitektur sesuai spesifikasi. Config menggunakan `96x96`, batch `64`, tepat `20` epoch tanpa early stopping, dan otomatis memakai GPU kalau TensorFlow mendeteksi GPU.


In [ ]:
grid = generate_shared_conv_grid(model_config)
print(f"Total experiments: {len(grid)}")

for index, config in enumerate(grid, start=1):
    print(
        f"{index:02d}. {config.name} | "
        f"filters={config.conv_filters} | kernels={config.kernel_sizes} | pooling={config.pooling_type}"
    )

In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    run_grid(model_config, training_config, skip_completed=True)
else:
    print("Training skipped. Set RUN_TRAINING = True to train all 16 CNN models.")


## Ringkasan Hasil Bagian 3

Cell ini membaca metadata hasil training dari `artifacts/experiments/cnn`.

In [ ]:
history_dir = PROJECT_ROOT / training_config.history_dir
metadata_files = sorted(history_dir.glob("*.json"))

runs = []
for path in metadata_files:
    metadata = json.loads(path.read_text(encoding="utf-8"))
    cfg = metadata["model_config"]
    metrics = metadata.get("metrics", {})
    history = metadata.get("history", {})
    runs.append({"path": path, "config": cfg, "metrics": metrics, "history": history, "metadata": metadata})

if not runs:
    print(f"No training metadata found in {history_dir}.")
else:
    for run in sorted(runs, key=lambda item: item["metrics"].get("validation_macro_f1", -1), reverse=True):
        cfg = run["config"]
        print(
            cfg["name"],
            "| val_macro_f1=", round(run["metrics"].get("validation_macro_f1", 0.0), 4),
            "| best_val_loss=", round(min(run["history"].get("val_loss", [0.0])), 4),
        )


In [ ]:
if runs:
    plt.figure(figsize=(12, 6))
    for run in runs:
        history = run["history"]
        name = run["config"]["name"]
        if "val_loss" in history:
            plt.plot(history["val_loss"], label=name)
    plt.title("Validation Loss untuk 16 Arsitektur CNN")
    plt.xlabel("Epoch")
    plt.ylabel("Validation Loss")
    plt.legend(fontsize=7, ncol=2)
    plt.show()

## Analisis Hyperparameter

Cell ini merangkum pengaruh jumlah layer, kombinasi filter, ukuran kernel, dan jenis pooling berdasarkan rata-rata validation macro F1 dari metadata training.


In [ ]:
def summarize_by(key_fn, label):
    groups = {}
    for run in runs:
        key = key_fn(run["config"])
        groups.setdefault(key, []).append(run["metrics"].get("validation_macro_f1", 0.0))

    print(label)
    for key, values in sorted(groups.items(), key=lambda item: str(item[0])):
        print(f"  {key}: mean_val_macro_f1={np.mean(values):.4f} over {len(values)} runs")
    print()

if runs:
    summarize_by(lambda cfg: len(cfg["conv_filters"]), "Jumlah layer konvolusi")
    summarize_by(lambda cfg: tuple(cfg["conv_filters"]), "Kombinasi filter")
    summarize_by(lambda cfg: tuple(cfg["kernel_sizes"]), "Ukuran kernel")
    summarize_by(lambda cfg: cfg["pooling_type"], "Jenis pooling")


## Bagian 4: Eksperimen dan Evaluasi

Bagian ini memilih arsitektur terbaik dari Bagian 3 berdasarkan validation macro F1-score, lalu membandingkan Keras dan forward propagation scratch pada split test.

In [ ]:
if not runs:
    raise RuntimeError("Run Bagian 3 training first before Bagian 4 evaluation.")

SCRATCH_EVAL_BATCHES = None  # full test split; set angka kecil untuk debugging cepat

best_run = max(runs, key=lambda item: item["metrics"].get("validation_macro_f1", -1))
best_metadata = best_run["metadata"]
best_name = best_run["config"]["name"]
model_path = PROJECT_ROOT / best_metadata["artifacts"]["model_path"]

print("Best model:", best_name)
print("Model path:", model_path)

keras_model = tf.keras.models.load_model(model_path)



In [ ]:
test_ds, class_names = build_intel_test_dataset(
    test_dir=PROJECT_ROOT / "data/raw/intel_image_classification/seg_test/seg_test",
    image_size=training_config.image_size,
    batch_size=training_config.batch_size,
)

keras_metrics = evaluate_keras_model(keras_model, test_ds, num_classes=len(class_names))
keras_metrics


In [ ]:
scratch_shared = build_scratch_cnn_from_keras(
    keras_model,
    replace_conv_with_local=False,
    input_shape=tuple(best_run["config"]["input_shape"]),
)

scratch_shared_metrics = evaluate_scratch_model(
    scratch_shared,
    test_ds,
    num_classes=len(class_names),
    max_batches=SCRATCH_EVAL_BATCHES,
)
scratch_shared_metrics


In [ ]:
scratch_non_shared = build_scratch_cnn_from_keras(
    keras_model,
    replace_conv_with_local=True,
    input_shape=tuple(best_run["config"]["input_shape"]),
)

scratch_non_shared_metrics = evaluate_scratch_model(
    scratch_non_shared,
    test_ds,
    num_classes=len(class_names),
    max_batches=SCRATCH_EVAL_BATCHES,
)

comparison = {
    "keras_shared_macro_f1_full_test": keras_metrics["macro_f1"],
    "scratch_shared_macro_f1_limited": scratch_shared_metrics["macro_f1"],
    "scratch_non_shared_macro_f1_limited": scratch_non_shared_metrics["macro_f1"],
    "scratch_eval_batches": SCRATCH_EVAL_BATCHES,
    "keras_parameter_count": keras_model.count_params(),
    "scratch_shared_parameter_count": scratch_shared.count_parameters(),
    "scratch_non_shared_parameter_count": scratch_non_shared.count_parameters(),
}
comparison


## Bonus: Visualisasi Feature Maps dan Grad-CAM

Cell berikut menyimpan visualisasi intermediate feature maps dari layer Conv2D dan Grad-CAM untuk satu gambar test.

In [ ]:
sample_batch, sample_labels = next(iter(test_ds.take(1)))
sample_image = sample_batch[0].numpy()
sample_label = int(sample_labels[0].numpy())

plt.figure(figsize=(4, 4))
plt.imshow(sample_image)
plt.title(f"Ground truth: {class_names[sample_label]}")
plt.axis("off")
plt.show()

In [ ]:
feature_output_dir = PROJECT_ROOT / "artifacts/plots/cnn/feature_maps"
feature_paths = save_conv_feature_visualizations(
    model=keras_model,
    images=sample_image,
    output_dir=feature_output_dir,
    max_channels=16,
)

feature_paths

In [ ]:
predicted_class = int(tf.argmax(keras_model.predict(sample_image[None, ...], verbose=0)[0]).numpy())
gradcam_path = PROJECT_ROOT / "artifacts/plots/cnn/grad_cam" / f"{best_name}_sample_gradcam.png"

heatmap = save_gradcam(
    model=keras_model,
    image=sample_image,
    output_path=gradcam_path,
    class_index=predicted_class,
)

print("Predicted class:", class_names[predicted_class])
print("Grad-CAM saved to:", gradcam_path)
plt.imshow(heatmap, cmap="jet")
plt.axis("off")
plt.show()